# 05 - Build Candidate Profile

Notebook này gom các kết quả từ các bước trước để tạo **candidate profile**.

Hiểu đơn giản:

```text
CV sạch + section CV + skill đã tìm thấy + skill đã map taxonomy
        ↓
hồ sơ ứng viên đã xử lý
        ↓
05_candidate_profiles.xlsx
```

File này chưa match với job.  
File này chỉ tạo “hồ sơ nghề nghiệp” của từng ứng viên để bước 06 đem đi so với job.

## 1. Import thư viện

- `pandas`: xử lý bảng dữ liệu.
- `ast`: đọc list nếu list bị lưu thành chuỗi trong Excel.
- `Path`: quản lý đường dẫn.

In [16]:
import ast
from pathlib import Path

import pandas as pd

## 2. Khai báo đường dẫn

Notebook này dùng 4 file đầu vào:

- `01_cv_cleaned.xlsx`: CV đã làm sạch text.
- `02_cv_sectioned.xlsx`: CV đã tách section.
- `03_cv_skills_extracted.xlsx`: skill đã trích xuất từ CV.
- `04_cv_skills_mapped.xlsx`: skill đã được map vào taxonomy.

Output:

- `05_candidate_profiles.xlsx`: hồ sơ ứng viên đã xử lý.

In [17]:
BASE_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline")

CLEAN_PATH = BASE_DIR / "data_outputs" / "step01_clean_text" / "01_cv_cleaned.xlsx"
SECTION_PATH = BASE_DIR / "data_outputs" / "step02_sections" / "02_cv_sectioned.xlsx"
EXTRACT_PATH = BASE_DIR / "data_outputs" / "step03_skill_extract" / "03_cv_skills_extracted.xlsx"
MAP_PATH = BASE_DIR / "data_outputs" / "step04_skill_map" / "04_cv_skills_mapped.xlsx"

OUTPUT_DIR = BASE_DIR / "data_outputs" / "step05_candidate_profile"
OUTPUT_PATH = OUTPUT_DIR / "05_candidate_profiles.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CLEAN_PATH:", CLEAN_PATH, CLEAN_PATH.exists())
print("SECTION_PATH:", SECTION_PATH, SECTION_PATH.exists())
print("EXTRACT_PATH:", EXTRACT_PATH, EXTRACT_PATH.exists())
print("MAP_PATH:", MAP_PATH, MAP_PATH.exists())
print("OUTPUT_PATH:", OUTPUT_PATH)

CLEAN_PATH: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step01_clean_text/01_cv_cleaned.xlsx True
SECTION_PATH: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step02_sections/02_cv_sectioned.xlsx True
EXTRACT_PATH: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step03_skill_extract/03_cv_skills_extracted.xlsx True
MAP_PATH: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step04_skill_map/04_cv_skills_mapped.xlsx True
OUTPUT_PATH: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step05_candidate_profile/05_candidate_profiles.xlsx


## 3. Đọc dữ liệu

Đọc 4 bảng đầu vào.

Mầm non recommendation:

```text
clean_df   = CV đã lau sạch
section_df = CV đã chia ngăn
extract_df = skill tìm được trong CV
map_df     = skill đã tra hộ khẩu taxonomy
```

In [18]:
clean_df = pd.read_excel(CLEAN_PATH, engine="openpyxl")
section_df = pd.read_excel(SECTION_PATH, engine="openpyxl")
extract_df = pd.read_excel(EXTRACT_PATH, engine="openpyxl")
map_df = pd.read_excel(MAP_PATH, engine="openpyxl")

print("clean_df:", clean_df.shape)
print("section_df:", section_df.shape)
print("extract_df:", extract_df.shape)
print("map_df:", map_df.shape)

display(clean_df.head(2))
display(section_df.head(2))
display(extract_df.head(2))
display(map_df.head(2))

clean_df: (20, 6)
section_df: (20, 20)
extract_df: (20, 28)
map_df: (38, 9)


,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False


,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean,section_summary,section_skills,section_experience,section_education,section_projects,section_certifications,section_other,section_summary_len,section_skills_len,section_experience_len,section_education_len,section_projects_len,section_certifications_len,section_other_len
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False,NaN,NaN,NaN,NaN,NaN,NaN,frontend developer experienced in reactjs vuej...,0,0,0,0,0,0,240
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False,NaN,NaN,NaN,NaN,NaN,NaN,backend engineer experienced in java spring bo...,0,0,0,0,0,0,181


,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean,section_summary,section_skills,section_experience,section_education,...,section_certifications_len,section_other_len,matched_skills_skills,matched_skills_experience,matched_skills_projects,matched_skills_other,matched_skills_summary,matched_skills_clean_text,matched_skills_all,n_matched_skills
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False,NaN,NaN,NaN,NaN,...,0,240,[],[],[],"['javascript', 'implement frontend website des...",[],"['javascript', 'implement frontend website des...","['javascript', 'implement frontend website des...",3
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False,NaN,NaN,NaN,NaN,...,0,181,[],[],[],"['perform software unit testing', 'mysql']",[],"['perform software unit testing', 'mysql']","['perform software unit testing', 'mysql']",2


,candidate_id,raw_skill,mapped_skill,mapping_source,mapping_confidence,nhom_lon,nhom_nho,cum_ky_nang,skill_subgroup
0,C001,javascript,JavaScript,exact_clean,1,Software Development,Frontend,Programming Languages,Web Development
1,C001,implement frontend website design,implement frontend website design,unmapped,0,NaN,NaN,NaN,NaN


## 4. Kiểm tra cột bắt buộc

Trước khi xử lý, kiểm tra các file có đủ cột cần thiết không.

Nếu thiếu cột thì dừng ngay, tránh chạy sai âm thầm.

In [19]:
required_clean_cols = ["candidate_id", "cv_text_clean"]

required_section_cols = [
    "candidate_id",
    "section_summary",
    "section_experience",
    "section_projects",
    "section_other",
]

required_extract_cols = ["candidate_id", "matched_skills_all"]

required_map_cols = [
    "candidate_id",
    "mapped_skill",
    "mapping_source",
    "nhom_lon",
    "skill_subgroup",
]


def check_required_cols(df, required_cols, df_name):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} thiếu cột: {missing}")


check_required_cols(clean_df, required_clean_cols, "clean_df")
check_required_cols(section_df, required_section_cols, "section_df")
check_required_cols(extract_df, required_extract_cols, "extract_df")
check_required_cols(map_df, required_map_cols, "map_df")

print("Tất cả input đủ cột cần thiết.")

Tất cả input đủ cột cần thiết.


## 5. Hàm parse list

Một số cột trong Excel nhìn giống list nhưng thực chất là chuỗi text.

Ví dụ Excel lưu:

```text
["mysql", "react"]
```

Hàm này biến nó thành list Python thật:

```python
["mysql", "react"]
```

In [20]:
def parse_list(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    value = str(value).strip()
    if not value:
        return []

    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    return [value]

## 6. Lấy skill thô đã phát hiện từ file 03

File 03 có cột `matched_skills_all`.

Bước này đổi nó thành:

- `skills_raw_detected`: danh sách skill thô đã tìm thấy.
- `n_raw_skills`: số lượng skill thô.

Ví dụ:

```text
matched_skills_all = ["mysql", "react"]
→ skills_raw_detected = ["mysql", "react"]
→ n_raw_skills = 2
```

In [21]:
extract_df = extract_df.copy()

extract_df["skills_raw_detected"] = extract_df["matched_skills_all"].apply(parse_list)
extract_df["n_raw_skills"] = extract_df["skills_raw_detected"].apply(len)

display(extract_df[["candidate_id", "skills_raw_detected", "n_raw_skills"]].head())

,candidate_id,skills_raw_detected,n_raw_skills
0,C001,"[javascript, implement frontend website design...",3
1,C002,"[perform software unit testing, mysql]",2
2,C003,"[perform data analysis, sql]",2
3,C004,"[web services, devops]",2
4,C005,[],0


## 7. Chỉ giữ skill đã map thành công

File 04 có thể có skill chưa map được.

Bước này chỉ giữ các skill có `mapping_source` đáng tin:

- `exact_clean`: match trực tiếp với file 14.
- `djinni_alias`: match qua alias rồi về file 14.
- `djinni_only`: match qua nguồn Djinni nếu có.

Skill `unmapped` bị loại ở bước này.

In [22]:
valid_mapping_sources = ["exact_clean", "djinni_alias", "djinni_only"]

mapped_only_df = map_df[
    map_df["mapping_source"].isin(valid_mapping_sources)
].copy()

print("mapped_only_df:", mapped_only_df.shape)
display(mapped_only_df.head())

mapped_only_df: (31, 9)


,candidate_id,raw_skill,mapped_skill,mapping_source,mapping_confidence,nhom_lon,nhom_nho,cum_ky_nang,skill_subgroup
0,C001,javascript,JavaScript,exact_clean,1,Software Development,Frontend,Programming Languages,Web Development
2,C001,css,CSS,exact_clean,1,Software Development,Frontend,Programming Languages,Web Development
3,C002,perform software unit testing,perform software unit testing,exact_clean,1,Software Development,Code Quality & Review,Software Development,Software Testing
4,C002,mysql,MySQL,exact_clean,1,Software Development,Backend,Databases,Database Management
5,C003,perform data analysis,perform data analysis,exact_clean,1,Data & Databases,Data Analysis,Data & Analytics,Data Analysis


## 8. Gom skill theo từng candidate

File 04 có dạng:

```text
candidate_01 - MySQL
candidate_01 - React
candidate_01 - TypeScript
```

Bước này gom lại thành:

```text
candidate_01:
- mapped_skills = [MySQL, React, TypeScript]
- skill_groups = [...]
- skill_subgroups = [...]
```

In [23]:
def unique_preserve_order(values):
    seen = set()
    result = []

    for value in values:
        if pd.isna(value):
            continue

        value = str(value).strip()
        if not value:
            continue

        if value not in seen:
            seen.add(value)
            result.append(value)

    return result


mapped_agg_df = (
    mapped_only_df
    .groupby("candidate_id", as_index=False)
    .agg({
        "mapped_skill": lambda x: unique_preserve_order(x),
        "nhom_lon": lambda x: unique_preserve_order(x),
        "skill_subgroup": lambda x: unique_preserve_order(x),
    })
)

mapped_agg_df = mapped_agg_df.rename(columns={
    "mapped_skill": "mapped_skills",
    "nhom_lon": "skill_groups",
    "skill_subgroup": "skill_subgroups",
})

mapped_agg_df["n_mapped_skills"] = mapped_agg_df["mapped_skills"].apply(len)

display(mapped_agg_df.head())

,candidate_id,mapped_skills,skill_groups,skill_subgroups,n_mapped_skills
0,C001,"[JavaScript, CSS]",[Software Development],[Web Development],2
1,C002,"[perform software unit testing, MySQL]",[Software Development],"[Software Testing, Database Management]",2
2,C003,"[perform data analysis, SQL]",[Data & Databases],"[Data Analysis, Query Languages & Analytics]",2
3,C004,"[web services, DevOps]",[IT Infrastructure & Operations],"[Web Services / APIs, DevOps]",2
4,C007,[machine learning],[Data & AI],[Machine Learning / AI],1


## 9. Tính dominant group

`dominant_group` là nhóm chính/nhóm trội của ứng viên.

Cách làm đơn giản:

```text
nhóm nào xuất hiện nhiều nhất trong các skill đã map
→ nhóm đó là dominant_group
```

Ví dụ:

```text
React      → Software Development
TypeScript → Software Development
MySQL      → Data & AI

Software Development xuất hiện nhiều nhất
→ dominant_group = Software Development
```

In [24]:
# Ưu tiên dùng cột thể hiện taxonomy group thật.
# Trong file 04 hiện tại, cột này đang tên là "nhom_lon".
TAXONOMY_GROUP_COL = "nhom_lon"

group_count_df = (
    mapped_only_df[
        mapped_only_df[TAXONOMY_GROUP_COL].fillna("").astype(str).str.strip() != ""
    ]
    .groupby(["candidate_id", TAXONOMY_GROUP_COL])
    .size()
    .reset_index(name="count")
)

dominant_group_df = (
    group_count_df
    .sort_values(["candidate_id", "count"], ascending=[True, False])
    .drop_duplicates(subset=["candidate_id"])
    [["candidate_id", TAXONOMY_GROUP_COL]]
    .rename(columns={TAXONOMY_GROUP_COL: "dominant_group"})
)

display(dominant_group_df.head())

,candidate_id,dominant_group
0,C001,Software Development
1,C002,Software Development
2,C003,Data & Databases
3,C004,IT Infrastructure & Operations
4,C007,Data & AI


## 10. Ghép tất cả lại thành candidate profile

Bước này merge các bảng theo `candidate_id`.

Công thức mầm non:

```text
CV sạch
+ section CV
+ skill thô
+ skill đã map
+ dominant group
= candidate profile
```

In [25]:
profile_df = clean_df[["candidate_id", "cv_text_clean"]].copy()

profile_df = profile_df.merge(
    section_df[[
        "candidate_id",
        "section_summary",
        "section_experience",
        "section_projects",
        "section_other",
    ]],
    on="candidate_id",
    how="left",
)

profile_df = profile_df.merge(
    extract_df[["candidate_id", "skills_raw_detected", "n_raw_skills"]],
    on="candidate_id",
    how="left",
)

profile_df = profile_df.merge(
    mapped_agg_df,
    on="candidate_id",
    how="left",
)

profile_df = profile_df.merge(
    dominant_group_df,
    on="candidate_id",
    how="left",
)

display(profile_df.head())

,candidate_id,cv_text_clean,section_summary,section_experience,section_projects,section_other,skills_raw_detected,n_raw_skills,mapped_skills,skill_groups,skill_subgroups,n_mapped_skills,dominant_group
0,C001,frontend developer experienced in reactjs vuej...,NaN,NaN,NaN,frontend developer experienced in reactjs vuej...,"[javascript, implement frontend website design...",3,"[JavaScript, CSS]",[Software Development],[Web Development],2.0,Software Development
1,C002,backend engineer experienced in java spring bo...,NaN,NaN,NaN,backend engineer experienced in java spring bo...,"[perform software unit testing, mysql]",2,"[perform software unit testing, MySQL]",[Software Development],"[Software Testing, Database Management]",2.0,Software Development
2,C003,data analyst experienced in python pandas nump...,NaN,NaN,NaN,data analyst experienced in python pandas nump...,"[perform data analysis, sql]",2,"[perform data analysis, SQL]",[Data & Databases],"[Data Analysis, Query Languages & Analytics]",2.0,Data & Databases
3,C004,devops engineer experienced in linux docker je...,NaN,NaN,NaN,devops engineer experienced in linux docker je...,"[web services, devops]",2,"[web services, DevOps]",[IT Infrastructure & Operations],"[Web Services / APIs, DevOps]",2.0,IT Infrastructure & Operations
4,C005,qa engineer experienced in manual testing test...,NaN,NaN,NaN,qa engineer experienced in manual testing test...,[],0,NaN,NaN,NaN,NaN,NaN


## 11. Dọn dữ liệu rỗng

Sau khi merge, có candidate có thể không có skill hoặc không có group.

Bước này chuẩn hóa:

- cột list rỗng → `[]`
- số lượng skill rỗng → `0`
- dominant group rỗng → chuỗi rỗng

In [26]:
list_cols = [
    "skills_raw_detected",
    "mapped_skills",
    "skill_groups",
    "skill_subgroups",
]

for col in list_cols:
    profile_df[col] = profile_df[col].apply(
        lambda x: x if isinstance(x, list) else []
    )

profile_df["n_raw_skills"] = profile_df["n_raw_skills"].fillna(0).astype(int)
profile_df["n_mapped_skills"] = profile_df["n_mapped_skills"].fillna(0).astype(int)
profile_df["dominant_group"] = profile_df["dominant_group"].fillna("")

display(profile_df.head())

,candidate_id,cv_text_clean,section_summary,section_experience,section_projects,section_other,skills_raw_detected,n_raw_skills,mapped_skills,skill_groups,skill_subgroups,n_mapped_skills,dominant_group
0,C001,frontend developer experienced in reactjs vuej...,NaN,NaN,NaN,frontend developer experienced in reactjs vuej...,"[javascript, implement frontend website design...",3,"[JavaScript, CSS]",[Software Development],[Web Development],2,Software Development
1,C002,backend engineer experienced in java spring bo...,NaN,NaN,NaN,backend engineer experienced in java spring bo...,"[perform software unit testing, mysql]",2,"[perform software unit testing, MySQL]",[Software Development],"[Software Testing, Database Management]",2,Software Development
2,C003,data analyst experienced in python pandas nump...,NaN,NaN,NaN,data analyst experienced in python pandas nump...,"[perform data analysis, sql]",2,"[perform data analysis, SQL]",[Data & Databases],"[Data Analysis, Query Languages & Analytics]",2,Data & Databases
3,C004,devops engineer experienced in linux docker je...,NaN,NaN,NaN,devops engineer experienced in linux docker je...,"[web services, devops]",2,"[web services, DevOps]",[IT Infrastructure & Operations],"[Web Services / APIs, DevOps]",2,IT Infrastructure & Operations
4,C005,qa engineer experienced in manual testing test...,NaN,NaN,NaN,qa engineer experienced in manual testing test...,[],0,[],[],[],0,


## 12. Đổi tên cột text profile

Đổi `cv_text_clean` thành `profile_text_clean`.

Tên mới thể hiện rằng đây là text đại diện cho profile ứng viên.

In [27]:
profile_df = profile_df.rename(columns={
    "cv_text_clean": "profile_text_clean"
})

## 13. Chọn các cột cuối cùng

Đây là bảng candidate profile cuối cùng.

Các cột quan trọng:

- `profile_text_clean`: text CV đã clean.
- `skills_raw_detected`: skill thô tìm được ở file 03.
- `mapped_skills`: skill đã chuẩn hóa/map ở file 04.
- `skill_groups`: nhóm taxonomy của skill.
- `dominant_group`: nhóm chính của ứng viên.

In [28]:
final_cols = [
    "candidate_id",
    "profile_text_clean",
    "skills_raw_detected",
    "mapped_skills",
    "skill_groups",
    "skill_subgroups",
    "n_raw_skills",
    "n_mapped_skills",
    "dominant_group",
    "section_summary",
    "section_experience",
    "section_projects",
    "section_other",
]

profile_df = profile_df[final_cols]

print("profile_df shape:", profile_df.shape)
display(profile_df.head())

profile_df shape: (20, 13)


,candidate_id,profile_text_clean,skills_raw_detected,mapped_skills,skill_groups,skill_subgroups,n_raw_skills,n_mapped_skills,dominant_group,section_summary,section_experience,section_projects,section_other
0,C001,frontend developer experienced in reactjs vuej...,"[javascript, implement frontend website design...","[JavaScript, CSS]",[Software Development],[Web Development],3,2,Software Development,NaN,NaN,NaN,frontend developer experienced in reactjs vuej...
1,C002,backend engineer experienced in java spring bo...,"[perform software unit testing, mysql]","[perform software unit testing, MySQL]",[Software Development],"[Software Testing, Database Management]",2,2,Software Development,NaN,NaN,NaN,backend engineer experienced in java spring bo...
2,C003,data analyst experienced in python pandas nump...,"[perform data analysis, sql]","[perform data analysis, SQL]",[Data & Databases],"[Data Analysis, Query Languages & Analytics]",2,2,Data & Databases,NaN,NaN,NaN,data analyst experienced in python pandas nump...
3,C004,devops engineer experienced in linux docker je...,"[web services, devops]","[web services, DevOps]",[IT Infrastructure & Operations],"[Web Services / APIs, DevOps]",2,2,IT Infrastructure & Operations,NaN,NaN,NaN,devops engineer experienced in linux docker je...
4,C005,qa engineer experienced in manual testing test...,[],[],[],[],0,0,,NaN,NaN,NaN,qa engineer experienced in manual testing test...


## 14. Lưu output

Xuất file:

```text
05_candidate_profiles.xlsx
```

File này là input cho bước 06.

In [29]:
profile_df.to_excel(OUTPUT_PATH, index=False)
print("Đã lưu file:", OUTPUT_PATH)

Đã lưu file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step05_candidate_profile/05_candidate_profiles.xlsx


## 15. Đọc lại output để kiểm tra

Đọc lại file vừa lưu để chắc chắn output đã ghi thành công.

In [30]:
result = pd.read_excel(OUTPUT_PATH, engine="openpyxl")

print("Output shape:", result.shape)
display(result.head())

Output shape: (20, 13)


,candidate_id,profile_text_clean,skills_raw_detected,mapped_skills,skill_groups,skill_subgroups,n_raw_skills,n_mapped_skills,dominant_group,section_summary,section_experience,section_projects,section_other
0,C001,frontend developer experienced in reactjs vuej...,"['javascript', 'implement frontend website des...","['JavaScript', 'CSS']",['Software Development'],['Web Development'],3,2,Software Development,NaN,NaN,NaN,frontend developer experienced in reactjs vuej...
1,C002,backend engineer experienced in java spring bo...,"['perform software unit testing', 'mysql']","['perform software unit testing', 'MySQL']",['Software Development'],"['Software Testing', 'Database Management']",2,2,Software Development,NaN,NaN,NaN,backend engineer experienced in java spring bo...
2,C003,data analyst experienced in python pandas nump...,"['perform data analysis', 'sql']","['perform data analysis', 'SQL']",['Data & Databases'],"['Data Analysis', 'Query Languages & Analytics']",2,2,Data & Databases,NaN,NaN,NaN,data analyst experienced in python pandas nump...
3,C004,devops engineer experienced in linux docker je...,"['web services', 'devops']","['web services', 'DevOps']",['IT Infrastructure & Operations'],"['Web Services / APIs', 'DevOps']",2,2,IT Infrastructure & Operations,NaN,NaN,NaN,devops engineer experienced in linux docker je...
4,C005,qa engineer experienced in manual testing test...,[],[],[],[],0,0,NaN,NaN,NaN,NaN,qa engineer experienced in manual testing test...


## Tóm tắt file 05

```text
01_cv_cleaned.xlsx
+ 02_cv_sectioned.xlsx
+ 03_cv_skills_extracted.xlsx
+ 04_cv_skills_mapped.xlsx
        ↓
05_candidate_profiles.xlsx
```

Nói ngắn gọn:

> File 05 gom toàn bộ thông tin đã xử lý của CV để tạo hồ sơ ứng viên có cấu trúc.